In [177]:
import re,numpy as np,pandas as pd
from collections import Counter
import torch,torch.nn as nn
from torch.utils.data import Dataset,DataLoader,TensorDataset
from sklearn.metrics import f1_score

MAX_VOCAB,MAX_LEN,EMBED_DIM,BATCH,EPOCHS,LR=10000,100,64,64,10,1e-3
DEVICE="cuda" if torch.cuda.is_available() else "cpu"

train_df=pd.read_csv("train.csv")
test_df=pd.read_csv("test.csv")

def clean(s):
  return re.sub(r"[^a-z\s]"," ",str(s).lower()).split()

all_tokens=[tok for token in train_df['customer_review'].apply(clean) for tok in token]
vocab={"<pad>":0,"<unk>":1}
for w,_ in Counter(all_tokens).most_common(MAX_VOCAB-2):
  vocab[w]=len(vocab)

def encode(text):
  ids=[vocab.get(t,1) for t in clean(text)[:MAX_LEN]]
  return ids+[0]*(MAX_LEN-len(ids))

X=torch.tensor([encode(t) for t in train_df["customer_review"]],dtype=torch.long)
y=torch.tensor(train_df["feedback"].values,dtype=torch.float)
X_test=torch.tensor([encode(t) for t in test_df["customer_review"]],dtype=torch.long)

split=int(0.85*len(X))
train_loader=DataLoader(TensorDataset(X[:split], y[:split]),batch_size=BATCH,shuffle=True)
val_loader=DataLoader(TensorDataset(X[split:], y[split:]),batch_size=BATCH)
test_loader=DataLoader(X_test,batch_size=BATCH)

class TextClassifier(nn.Module):
  def __init__(self):
    super().__init__()
    self.embed=nn.Embedding(len(vocab),EMBED_DIM,padding_idx=0)
    self.fc=nn.Sequential(
        nn.Linear(EMBED_DIM,64),
        nn.ReLU(),
        nn.Linear(64,1)
    )
  def forward(self,x):
    x=self.embed(x).mean(dim=1)
    return self.fc(x).squeeze(1)

model=TextClassifier().to(DEVICE)
optimizer=torch.optim.Adam(model.parameters(),lr=LR)
criterion=nn.BCEWithLogitsLoss()

for epoch in range(EPOCHS):
  model.train()
  for xb,yb in train_loader:
    xb,yb=xb.to(DEVICE),yb.to(DEVICE)
    optimizer.zero_grad()
    loss=criterion(model(xb),yb)
    loss.backward()
    optimizer.step()

  model.eval()
  preds,true=[],[]
  with torch.no_grad():
    for xb,yb in val_loader:
      xb=xb.to(DEVICE)
      probs=torch.sigmoid(model(xb)).cpu()
      preds+=(probs>=0.5).int().tolist()
      true+=yb.int().tolist()
  print(f1_score(true,preds))

model.eval()
preds=[]
with torch.no_grad():
  for xb in test_loader:
    xb=xb.to(DEVICE)
    probs=torch.sigmoid(model(xb)).cpu()
    preds+=(probs>=0.5).int().tolist()

pd.DataFrame({
    "customer_review":test_df["customer_review"],
    "feedback":preds
}).to_csv("sub.csv",index=False)



0.6127167630057804
0.6127167630057804
0.6127167630057804
0.6127167630057804
0.6127167630057804
0.6127167630057804
0.6127167630057804
0.6127167630057804
0.6424242424242425
0.75177304964539
